# 모듈 08: RAG — 검색 증강 생성

## 이 모듈에서 배울 것

1. **전체 RAG 파이프라인**: 로드 → 분할 → 임베딩 → 저장 → 검색 → 생성까지 6단계 완성
2. **`create_stuff_documents_chain`**: 검색된 문서를 프롬프트에 삽입하는 체인
3. **`create_retrieval_chain`**: Retriever와 LLM을 하나의 체인으로 연결
4. **`create_history_aware_retriever`**: 이전 대화를 고려한 검색
5. **`RunnableWithMessageHistory`**: 세션별 대화 이력 자동 관리

## 학습 목표

이 모듈을 완료하면 다음을 할 수 있습니다:

1. `create_stuff_documents_chain`과 `create_retrieval_chain`으로 기본 RAG 파이프라인을 구성하고, `result["input"]` / `result["context"]` / `result["answer"]` 구조를 이해한다.
2. `create_history_aware_retriever`로 이전 대화 기록을 고려한 질문 재구성을 구현한다.
3. `RunnableWithMessageHistory`로 세션별 대화 이력을 관리하고, 2턴 이상의 대화형 RAG를 실행한다.

## 전체 흐름 — RAG 파이프라인 6단계

```
+------------------+     +------------------+     +------------------+
|  1단계: 로드      |     |  2단계: 분할      |     |  3단계: 임베딩    |
|  TextLoader      | --> | RecursiveChar... | --> | Embeddings       |
|  PDF, Web, ...   |     | chunk_size=300   |     | 텍스트 → 벡터    |
+------------------+     +------------------+     +--------+---------+
                                                            |
+------------------+     +------------------+     +--------v---------+
|  6단계: 생성      |     |  5단계: 검색      |     |  4단계: 저장      |
|  LLM + 프롬프트  | <-- |  Retriever       | <-- |  Chroma          |
|  최종 답변 생성   |     |  관련 청크 반환   |     |  벡터 DB 저장     |
+------------------+     +------------------+     +------------------+
  ★ 모듈 08 ★               ★ 모듈 08 ★              모듈 07
```

모듈 07에서 1~4단계(로드·분할·임베딩·저장)를 배웠습니다.  
이 모듈은 **5단계(검색)**와 **6단계(생성)**를 추가해 RAG를 완성합니다.

## RAG란?

**비유: 전문가 + 참고 문헌 서랍**

의사에게 진단을 요청한다고 상상해 보세요.

```
일반 LLM (전문가만 있는 경우)
  → 훈련 당시 알고 있던 지식으로만 답변
  → 우리 회사의 내부 문서나 최신 정보는 모름

RAG (전문가 + 참고 문헌 서랍)
  → 질문을 받으면 먼저 서랍에서 관련 문서를 꺼냄 (Retriever)
  → 꺼낸 문서를 읽으며 답변 생성 (LLM)
  → 내부 문서와 최신 정보를 반영한 정확한 답변 가능
```

| 구성 요소 | 비유 | LangChain 클래스 |
|-----------|------|------------------|
| 참고 문헌 서랍 | 벡터 DB | `Chroma` |
| 서랍 정리사 | 임베딩 | `GoogleGenerativeAIEmbeddings` |
| 사서 | 검색기 | `Retriever` |
| 전문가 | 언어 모델 | `ChatGoogleGenerativeAI` |
| 질문지 + 참고자료 묶음 | 프롬프트 조립 | `create_stuff_documents_chain` |
| 전체 프로세스 | 파이프라인 | `create_retrieval_chain` |

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document

# .env 로드 (08_rag/ 기준 상위 디렉토리)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

# API 키 확인
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

# 임베딩 + LLM 임포트 확인
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_huggingface import HuggingFaceEmbeddings
    print('[OK] ChatGoogleGenerativeAI, HuggingFaceEmbeddings 임포트 성공')
except ImportError as e:
    print(f'[FAIL] {e}')

# 체인 모듈 임포트 확인
try:
    from langchain_chroma import Chroma
    from langchain_community.document_loaders import TextLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_community.chat_message_histories import ChatMessageHistory
    from langchain_core.runnables.history import RunnableWithMessageHistory
    print('[OK] LangChain 체인 모듈 임포트 성공')
except ImportError as e:
    print(f'[FAIL] {e}')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...


/Users/sonny/Desktop/Dev/10X 생산성 에이전트 군단 - 사내 챗봇부터 AI 숏츠 생성기까지 외주 수익화 올인원 패키지/LangChain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] ChatGoogleGenerativeAI, HuggingFaceEmbeddings 임포트 성공
[OK] LangChain 체인 모듈 임포트 성공


## RAG 준비 단계 — 모듈 06·07 복습

RAG를 시작하기 전에 벡터 저장소를 먼저 구성해야 합니다.  
이 과정은 모듈 06(문서 로딩)과 모듈 07(임베딩·벡터 저장소)에서 배운 내용입니다.

```
모듈 06에서 배운 것:
  TextLoader         → 파일을 Document 객체로 로드
  RecursiveCharacterTextSplitter → Document를 작은 청크로 분할

모듈 07에서 배운 것:
  GoogleGenerativeAIEmbeddings → 텍스트를 768차원 벡터로 변환
  Chroma.from_documents()      → Document + 벡터를 함께 저장
  vectorstore.as_retriever()   → LCEL 체인에 연결 가능한 검색기 생성
```

아래 셀에서 샘플 문서를 만들고, 벡터 저장소까지 준비합니다.  
이 단계가 완료되면 RAG 체인 구성으로 넘어갑니다.

In [2]:
# 샘플 문서 생성
sample_text = """인공지능(AI) 기술 동향 2024

생성형 AI의 부상
2024년은 생성형 AI가 산업 전반에 빠르게 확산된 해입니다. ChatGPT, Claude, Gemini 등의
대형 언어 모델이 기업의 업무 자동화에 적극 활용되고 있습니다. 특히 코드 생성, 문서 작성,
고객 서비스 분야에서 큰 성과를 보이고 있습니다.

RAG 기술의 중요성
RAG(Retrieval-Augmented Generation)는 기업 내부 문서를 AI와 결합하는 핵심 기술로
자리잡았습니다. 기존 LLM의 지식 한계를 극복하고, 최신 정보와 내부 데이터를 활용할 수
있게 해줍니다. 의료, 법률, 금융 분야에서 특히 주목받고 있습니다.

멀티모달 AI
텍스트뿐만 아니라 이미지, 오디오, 비디오를 함께 처리하는 멀티모달 AI가 발전하고 있습니다.
Google Gemini와 GPT-4V는 이미지를 이해하고 분석하는 능력을 갖추었습니다.

AI 에이전트
여러 도구를 사용해 복잡한 작업을 자율적으로 수행하는 AI 에이전트 기술이 주목받고 있습니다.
LangChain의 에이전트 기능을 활용해 웹 검색, 코드 실행, 데이터 분석 등을 자동화할 수 있습니다.
"""

data_dir = os.path.join(notebook_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
txt_path = os.path.join(data_dir, 'ai_trends.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(sample_text)
print(f'[OK] 샘플 문서 생성: {txt_path}')

# 로드 + 분할
loader = TextLoader(txt_path, encoding='utf-8')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'[OK] 문서 분할: {len(chunks)}개 청크')

# 임베딩 (로컬 HuggingFace — API 호출 없이 안정적으로 동작)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.3)
rag_db_dir = os.path.join(notebook_dir, 'rag_db')
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory=rag_db_dir)
print(f'[OK] 벡터 저장소 생성: {vectorstore._collection.count()}개 벡터')

[OK] 샘플 문서 생성: /Users/sonny/Desktop/Dev/10X 생산성 에이전트 군단 - 사내 챗봇부터 AI 숏츠 생성기까지 외주 수익화 올인원 패키지/LangChain/08_rag/data/ai_trends.txt
[OK] 문서 분할: 3개 청크


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2796.78it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[OK] 벡터 저장소 생성: 6개 벡터


In [3]:
# Retriever 생성 + 미리 보기 테스트
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
test_docs = retriever.invoke('RAG 기술 활용 분야')

print(f'Retriever 테스트 — 검색된 청크 수: {len(test_docs)}개')
for i, doc in enumerate(test_docs, 1):
    print(f'  청크 {i}: {doc.page_content[:60]}...')
print('[OK] Retriever 준비 완료')

Retriever 테스트 — 검색된 청크 수: 3개
  청크 1: RAG 기술의 중요성
RAG(Retrieval-Augmented Generation)는 기업 내부 문서를 A...
  청크 2: RAG 기술의 중요성
RAG(Retrieval-Augmented Generation)는 기업 내부 문서를 A...
  청크 3: AI 에이전트
여러 도구를 사용해 복잡한 작업을 자율적으로 수행하는 AI 에이전트 기술이 주목받고 있습니다....
[OK] Retriever 준비 완료


## `create_stuff_documents_chain` — 문서를 프롬프트에 꽉꽉 채워 넣는 비서

**비유: 보고서 비서**

상사(LLM)에게 질문을 전달할 때, 비서가 관련 참고자료를 모아 함께 붙여주는 역할입니다.

```
입력:
  input   = '질문 텍스트'
  context = [Document1, Document2, Document3]  ← Retriever가 찾아준 문서들

create_stuff_documents_chain 처리:
  모든 Document.page_content를 하나로 합침 (stuff = 꽉꽉 채우기)
  프롬프트의 {context} 자리에 삽입

출력:
  LLM이 생성한 답변 텍스트
```

**왜 'stuff'인가?**  
검색된 모든 문서를 프롬프트에 **그냥 다 넣어버리는(stuff)** 가장 단순한 전략이기 때문입니다.  
문서가 너무 많으면 컨텍스트 한계를 초과할 수 있지만, 소규모 RAG에는 가장 효과적입니다.

In [4]:
# RAG 시스템 프롬프트 + create_stuff_documents_chain 생성
system_prompt = """당신은 문서 기반 질의응답 시스템입니다.
아래 컨텍스트를 바탕으로 질문에 답하세요.
컨텍스트에 없는 내용은 '제공된 문서에서 찾을 수 없습니다'라고 답하세요.

컨텍스트:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}'),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)

print('질문-답변 체인 구조:')
print('  입력: input (질문 텍스트) + context (List[Document])')
print('  출력: 답변 텍스트')
print('[OK] create_stuff_documents_chain 생성 성공')

질문-답변 체인 구조:
  입력: input (질문 텍스트) + context (List[Document])
  출력: 답변 텍스트
[OK] create_stuff_documents_chain 생성 성공


## `create_retrieval_chain` — 검색과 생성을 이어주는 파이프

**비유: 공장 생산 라인**

`create_retrieval_chain`은 두 단계를 하나의 파이프라인으로 연결합니다.

```
질문 입력
    |
    v
[Retriever] ← 첫 번째 공정: 질문과 관련된 문서 청크를 벡터 DB에서 꺼냄
    |
    v  (context = List[Document])
[create_stuff_documents_chain] ← 두 번째 공정: 문서를 프롬프트에 삽입 후 LLM 호출
    |
    v
result = {
    'input'  : '원래 질문',
    'context': [Document, ...],  ← 검색된 문서 청크
    'answer' : '최종 답변'       ← LLM이 생성한 답변
}
```

`invoke({'input': '질문'})` 하나만 호출하면 검색부터 생성까지 자동으로 처리됩니다.

In [5]:
# create_retrieval_chain 생성 + invoke
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

question = 'RAG 기술이 어떤 분야에서 주목받고 있나요?'
result = rag_chain.invoke({'input': question})

print(f'질문: {question}')
print()
print(f'답변: {result["answer"]}')
print()
print('result 딕셔너리 구조:')
print(f'  result["input"]   : {result["input"]}')
print(f'  result["context"] : List[Document] — {len(result["context"])}개 청크')
print(f'  result["answer"]  : 생성된 답변')
print('[OK] create_retrieval_chain + invoke 성공')

질문: RAG 기술이 어떤 분야에서 주목받고 있나요?

답변: RAG 기술은 의료, 법률, 금융 분야에서 특히 주목받고 있습니다.

result 딕셔너리 구조:
  result["input"]   : RAG 기술이 어떤 분야에서 주목받고 있나요?
  result["context"] : List[Document] — 3개 청크
  result["answer"]  : 생성된 답변
[OK] create_retrieval_chain + invoke 성공


In [6]:
# 여러 질문 테스트 — 문서 내 질문 2개 + 문서 외 질문 1개 (경계 테스트)
test_questions = [
    ('문서 내 질문', 'AI 에이전트로 무엇을 자동화할 수 있나요?'),
    ('문서 내 질문', '멀티모달 AI는 무엇을 처리할 수 있나요?'),
    ('문서 외 질문', '2024년 한국 경제 성장률은 얼마인가요?'),
]

for category, question in test_questions:
    result = rag_chain.invoke({'input': question})
    answer = result['answer']
    print(f'[{category}] {question}')
    print(f'  답변: {answer[:80]}...' if len(answer) > 80 else f'  답변: {answer}')
    print()

print('[OK] 문서 내/외 질문 경계 테스트 완료')

[문서 내 질문] AI 에이전트로 무엇을 자동화할 수 있나요?
  답변: AI 에이전트로는 웹 검색, 코드 실행, 데이터 분석 등을 자동화할 수 있습니다.

[문서 내 질문] 멀티모달 AI는 무엇을 처리할 수 있나요?
  답변: 제공된 문서에서 찾을 수 없습니다.

[문서 외 질문] 2024년 한국 경제 성장률은 얼마인가요?
  답변: 제공된 문서에서 찾을 수 없습니다.

[OK] 문서 내/외 질문 경계 테스트 완료


In [7]:
# result['context'] 분석 — 어떤 문서 청크가 검색됐나?
question = 'RAG 기술의 중요성은 무엇인가요?'
result = rag_chain.invoke({'input': question})

print(f'질문: "{question}"')
print(f'참조된 문서 청크 ({len(result["context"])}개):')
print('-' * 60)
for i, doc in enumerate(result['context'], 1):
    src = os.path.basename(doc.metadata.get('source', 'N/A'))
    print(f'청크 {i} (출처: {src}):')
    print(f'  {doc.page_content[:100]}...')
print('[OK] 검색된 컨텍스트 문서 분석 완료')

질문: "RAG 기술의 중요성은 무엇인가요?"
참조된 문서 청크 (3개):
------------------------------------------------------------
청크 1 (출처: ai_trends.txt):
  RAG 기술의 중요성
RAG(Retrieval-Augmented Generation)는 기업 내부 문서를 AI와 결합하는 핵심 기술로
자리잡았습니다. 기존 LLM의 지식 한계를 극...
청크 2 (출처: ai_trends.txt):
  RAG 기술의 중요성
RAG(Retrieval-Augmented Generation)는 기업 내부 문서를 AI와 결합하는 핵심 기술로
자리잡았습니다. 기존 LLM의 지식 한계를 극...
청크 3 (출처: ai_trends.txt):
  AI 에이전트
여러 도구를 사용해 복잡한 작업을 자율적으로 수행하는 AI 에이전트 기술이 주목받고 있습니다.
LangChain의 에이전트 기능을 활용해 웹 검색, 코드 실행, 데이...
[OK] 검색된 컨텍스트 문서 분석 완료


## 대화 인식 RAG — 이전 대화를 기억하는 검색

**문제: 기본 RAG의 한계**

```
사용자: RAG 기술이 어떤 분야에서 활용되나요?
AI:    의료, 법률, 금융 분야에서 주목받고 있습니다.

사용자: 이 기술의 한계는 무엇인가요?  ← '이 기술'이 RAG를 가리킴
AI:    (기본 RAG는 '이 기술'이 무엇인지 모름 → 엉뚱한 문서 검색)
```

**해결: `create_history_aware_retriever`**

```
사용자: 이 기술의 한계는 무엇인가요?
    |
    v  (LLM이 대화 기록 참조)
[질문 재구성] → 'RAG 기술의 한계는 무엇인가요?'  ← 독립적인 질문으로 변환
    |
    v
[Retriever]  → RAG 관련 문서 청크를 정확하게 검색
    |
    v
[LLM]        → 정확한 답변 생성
```

`RunnableWithMessageHistory`는 세션 ID별로 대화 이력을 자동으로 저장하고 불러옵니다.

In [8]:
# 1) 질문 재구성 프롬프트
contextualize_q_system_prompt = """이전 대화 기록과 최신 사용자 질문을 참고하여,
대화 기록 없이도 이해할 수 있는 독립적인 질문으로 재구성하세요.
질문에 답하지 말고, 필요하다면 질문을 재구성하고, 그렇지 않으면 그대로 반환하세요."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ('system', contextualize_q_system_prompt),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

# 2) 대화형 QA 프롬프트
qa_system_prompt = """당신은 문서 기반 질의응답 시스템입니다.
아래 컨텍스트를 바탕으로 질문에 답하세요.
컨텍스트에 없는 내용은 '제공된 문서에서 찾을 수 없습니다'라고 답하세요.

{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', qa_system_prompt),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

qa_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

# 3) 세션별 대화 이력 관리
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
    output_messages_key='answer',
)

print('[OK] 대화 인식 RAG 체인 구성 완료')
print('  create_history_aware_retriever: 이전 대화를 참고해 질문 재구성')
print('  RunnableWithMessageHistory: 세션별 대화 이력 자동 관리')

[OK] 대화 인식 RAG 체인 구성 완료
  create_history_aware_retriever: 이전 대화를 참고해 질문 재구성
  RunnableWithMessageHistory: 세션별 대화 이력 자동 관리


In [9]:
# 대화형 RAG 테스트 — 2턴 대화, 2번째 질문이 1번째 대화를 참조
session_id = 'session_001'
config = {'configurable': {'session_id': session_id}}

# 1번째 질문
q1 = 'RAG 기술이 어떤 분야에서 활용되나요?'
r1 = with_message_history.invoke({'input': q1}, config=config)
print(f'질문 1: {q1}')
print(f'답변 1: {r1["answer"]}')
print()

# 2번째 질문 — '이 기술'이 RAG를 가리킴 (이전 대화 참조 필요)
q2 = '이 기술의 한계는 무엇인가요?'
r2 = with_message_history.invoke({'input': q2}, config=config)
print(f'질문 2: {q2}  (← "이 기술" = RAG를 가리킴)')
print(f'답변 2: {r2["answer"]}')
print()

print(f'저장된 대화 이력: {len(store[session_id].messages)}개 메시지')
print('[OK] 대화 인식 RAG: 이전 대화 참조 성공')

질문 1: RAG 기술이 어떤 분야에서 활용되나요?
답변 1: RAG 기술은 의료, 법률, 금융 분야에서 특히 주목받고 있습니다.

질문 2: 이 기술의 한계는 무엇인가요?  (← "이 기술" = RAG를 가리킴)
답변 2: 제공된 문서에서 찾을 수 없습니다. (RAG 기술이 기존 LLM의 지식 한계를 극복한다고 언급되어 있지만, RAG 기술 자체의 한계에 대한 내용은 없습니다.)

저장된 대화 이력: 4개 메시지
[OK] 대화 인식 RAG: 이전 대화 참조 성공


## 전체 파이프라인 — 코드 대응 다이어그램

6단계 각각에 대응하는 LangChain 코드를 정리합니다.

```
단계        작업                     LangChain 코드
--------    ---------------------    ------------------------------------------
1. 로드      파일 → Document          TextLoader(path).load()
2. 분할      Document → 청크          RecursiveCharacterTextSplitter().split_documents()
3. 임베딩    텍스트 → 벡터            GoogleGenerativeAIEmbeddings(model=...)
4. 저장      청크 + 벡터 → DB         Chroma.from_documents(chunks, embeddings)
5. 검색      질문 → 관련 청크         vectorstore.as_retriever(search_kwargs={'k': 3})
6. 생성      청크 + 질문 → 답변       create_retrieval_chain(retriever, question_answer_chain)
```

**대화 인식 RAG 추가 구성:**

```
기존 Retriever
    |
    + create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
    |   → 이전 대화 참조 → 질문 재구성 → 검색
    |
    + RunnableWithMessageHistory(chain, get_session_history, ...)
        → 세션 ID별 대화 이력 자동 저장/불러오기
```

In [10]:
# 전체 RAG 파이프라인 one-shot — 로드 → 분할 → 저장 → 체인 → 질문
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print('=== 전체 RAG 파이프라인 ===')

# 1. 문서 로드
_txt = os.path.join(notebook_dir, 'data', 'ai_trends.txt')
_loader = TextLoader(_txt, encoding='utf-8')
_docs = _loader.load()
print(f'1단계 [로드]    : {len(_docs)}개 Document')

# 2. 분할
_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
_chunks = _splitter.split_documents(_docs)
print(f'2단계 [분할]    : {len(_chunks)}개 청크')

# 3. 임베딩 + 저장 (로컬 HuggingFace)
_embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
_vs = Chroma.from_documents(_chunks, _embeddings)
print(f'3단계 [임베딩·저장]: {_vs._collection.count()}개 벡터')

# 4. Retriever
_retriever = _vs.as_retriever(search_kwargs={'k': 3})
print(f'4단계 [Retriever]: k=3 설정')

# 5. 체인
_llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.3)
_prompt = ChatPromptTemplate.from_messages([
    ('system', '아래 컨텍스트로 답하세요. 없으면 모른다고 답하세요.\n\n{context}'),
    ('human', '{input}'),
])
_chain = create_retrieval_chain(_retriever, create_stuff_documents_chain(_llm, _prompt))

# 6. 질문
_q = '생성형 AI가 주로 활용되는 분야는?'
_r = _chain.invoke({'input': _q})
print(f'5단계 [검색·생성]: 질문 = "{_q}"')
answer_preview = _r['answer'][:80]
print(f'       답변 = {answer_preview}...')

print()
print('[OK] 전체 RAG 파이프라인 완료')

=== 전체 RAG 파이프라인 ===
1단계 [로드]    : 1개 Document
2단계 [분할]    : 3개 청크


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2397.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3단계 [임베딩·저장]: 3개 벡터
4단계 [Retriever]: k=3 설정
5단계 [검색·생성]: 질문 = "생성형 AI가 주로 활용되는 분야는?"
       답변 = 생성형 AI는 주로 코드 생성, 문서 작성, 고객 서비스 분야에서 활용됩니다....

[OK] 전체 RAG 파이프라인 완료


## 배운 것 정리

### 핵심 개념

| 개념 | 설명 | 비유 |
|------|------|------|
| RAG | 검색 + 생성 결합 | 전문가 + 참고 문헌 서랍 |
| `create_stuff_documents_chain` | 문서를 프롬프트에 삽입 | 보고서 비서 |
| `create_retrieval_chain` | Retriever + LLM 연결 | 공장 생산 라인 |
| `create_history_aware_retriever` | 대화 기록 반영 검색 | 문맥을 기억하는 사서 |
| `RunnableWithMessageHistory` | 세션별 이력 관리 | 대화 수첩 |

### 핵심 코드 패턴

```python
# 기본 RAG
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
result = rag_chain.invoke({'input': '질문'})
# result['input']   → 원래 질문
# result['context'] → List[Document] (검색된 청크)
# result['answer']  → LLM 생성 답변

# 대화 인식 RAG
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
    output_messages_key='answer',
)
result = with_message_history.invoke(
    {'input': '질문'},
    config={'configurable': {'session_id': 'session_001'}}
)
```

## 다음 모듈 예고: 모듈 09 — 에이전트

RAG로 문서를 검색하고 답변했다면, 에이전트는 한 단계 더 나아갑니다.

```
RAG (모듈 08):
  질문 → 문서 검색 → 답변 생성  (고정된 흐름)

에이전트 (모듈 09):
  질문 → 어떤 도구를 쓸지 스스로 판단 → 도구 실행 → 결과 확인 → 반복
  (동적인 흐름 — 웹 검색, 코드 실행, 계산기, RAG 등 여러 도구 조합)
```

### 모듈 09 준비 체크리스트

- [ ] 모듈 08 전체 셀 오류 없이 실행 완료
- [ ] `08_rag/rag_db/` 폴더가 생성되었는가?
- [ ] `08_rag/data/ai_trends.txt` 파일이 생성되었는가?
- [ ] 셀 16에서 2번째 질문 '이 기술'이 RAG로 해석된 답변이 출력되었는가?
- [ ] GOOGLE_API_KEY가 `.env`에 정상 설정되어 있는가?

### 모듈 09 핵심 코드 미리보기

```python
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    ...

tools = [search_web]
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
result = agent_executor.invoke({'input': '오늘 날씨는?'})
```

---

**수고하셨습니다!** 모듈 08 완료.